In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q17/q17_rewrite/checkpoints/pre_cell_2.pickle

In [4]:
%%RecordEvent
%%time
### cell 2 ###

# 1. Filter PART to get the relevant keys on GPU
target_keys = part[(part.P_BRAND == 'Brand#23') & (part.P_CONTAINER == 'MED BOX')].P_PARTKEY

# 2. Early‐filter LINEITEM to just those keys
li = lineitem[lineitem.L_PARTKEY.isin(target_keys)]

# 3. Compute the 20%‐scaled average quantity per key using transform (avoids a merge)
li['avg'] = li.groupby('L_PARTKEY').L_QUANTITY.transform('mean') * 0.2

# 4. Apply the quantity filter and compute the final metric entirely on GPU
good = li[li.L_QUANTITY < li.avg]
total = pd.DataFrame({'avg_yearly': [good.L_EXTENDEDPRICE.sum() / 7.0]})

CPU times: user 77.6 ms, sys: 24.6 ms, total: 102 ms
Wall time: 102 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q17/rewritten/o4_mini_high_small/checkpoints/post_cell_2_try_3.pickle